# 第39课 · 重写 1965 年改变世界的算法——从零递归实现 FFT（Cooley-Tukey），误差 < 1e-10

**目标**：手写 Cooley-Tukey 递归 FFT，让 `tests/audio/test_transforms.py` 全绿。

🔗 **Aurora 连接**：本节实现镜像 `src/aurora/audio/transforms.py` 的 `fft()` 函数。

← **上一课**　[L38 · FFT 蝶形分治](L38_fft_butterfly.ipynb)

> 上节课学习了 **FFT 蝶形分治**：偶奇拆分、蝶形运算 E[k]+W^k·O[k]，O(N²)→O(N log N)。  
> 本课将探讨 **从零手写 FFT**。

## 本课剧情：为什么"把大问题砍成两半"就能快 200 倍？

有道经典面试题：在一个有 1024 层的高塔里，找到某个楼层只需二分查找——每次砍半，最多 log₂1024=10 步。

FFT 做的正是同样的事情。DFT 需要对每个频点扫描全部 N 个样本——N 个频点 × N 个样本 = **N² 次乘法**。FFT 问："能不能只扫一半样本，然后把两个半结果合并成完整结果？"

**Cooley-Tukey 的答案**（1965）：

$$X[k] = \underbrace{\text{DFT}(x_{\text{even}})[k]}_{E[k]} + e^{-2\pi i k/N} \cdot \underbrace{\text{DFT}(x_{\text{odd}})[k]}_{O[k]}$$

把"偶数下标序列"和"奇数下标序列"各做 N/2 点 DFT，再用一次蝶形合并得到完整 N 点结果。这个分治**可以递归**：N/2 → N/4 → ... → 1（基础情形：单点 DFT 就是 x[0]）。

**效率**：log₂N 层 × N/2 蝶形/层 = **(N/2)·log₂N** 次乘法。

| N | DFT 乘法 | FFT 乘法 | 加速比 |
|---|---|---|---|
| 64 | 4096 | 192 | 21× |
| 1024 | 1048576 | 5120 | 205× |
| 4096 | 16777216 | 24576 | 682× |

本节任务：实现 `my_fft(x)` 递归 Cooley-Tukey FFT，并验证与 `np.fft.fft` 误差 < 1e-9。

In [ ]:
import numpy as np
import time
from aurora.audio.transforms import fft as aurora_fft

## 定位图：N=8 递归分治树

先看树的结构再动手写递归。向下是**拆分**（偶/奇），向上是**蝶形合并**；3 层对应 `log₂8=3` 次递归，每层 N/2=4 次蝶形乘法，总计 12 次——朴素 DFT 要 64 次。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4.2))

COLORS = {8: '#3498db', 4: '#e74c3c', 2: '#2ecc71', 1: '#95a5a6'}
LABELS = {8: 'my_fft(8)', 4: 'my_fft(4)', 2: 'my_fft(2)', 1: 'N=1\n返回原值'}
ys = [3.5, 2.4, 1.3, 0.2]   # y-centers per level
bh = 0.65                     # block height

def box(ax, cx, cy, w, n):
    ax.add_patch(plt.Rectangle((cx - w/2, cy - bh/2), w, bh,
                                fc=COLORS[n], ec='white', lw=2, alpha=0.88, zorder=3))
    ax.text(cx, cy, LABELS[n], ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', zorder=4)

# Boxes at each level
box(ax, 5.0, ys[0], 9.0, 8)
for cx in [2.5, 7.5]:
    box(ax, cx, ys[1], 4.1, 4)
for cx in [1.25, 3.75, 6.25, 8.75]:
    box(ax, cx, ys[2], 1.75, 2)
for cx in np.linspace(0.6, 9.4, 8):
    box(ax, cx, ys[3], 0.68, 1)

# Arrows (split direction, going down)
kw = dict(arrowstyle='->', color='#7f8c8d', lw=1.2)
for cx in [2.5, 7.5]:
    ax.annotate('', xy=(cx, ys[1] + bh/2 + 0.04),
                xytext=(5.0, ys[0] - bh/2 - 0.04), arrowprops=kw, zorder=2)
for parent, children in [(2.5, [1.25, 3.75]), (7.5, [6.25, 8.75])]:
    for cx in children:
        ax.annotate('', xy=(cx, ys[2] + bh/2 + 0.04),
                    xytext=(parent, ys[1] - bh/2 - 0.04), arrowprops=kw, zorder=2)
child_xs = np.linspace(0.6, 9.4, 8)
for i, parent in enumerate([1.25, 3.75, 6.25, 8.75]):
    for cx in child_xs[i * 2:(i + 1) * 2]:
        ax.annotate('', xy=(cx, ys[3] + bh/2 + 0.04),
                    xytext=(parent, ys[2] - bh/2 - 0.04), arrowprops=kw, zorder=2)

# Right-side annotations
for y, lbl in zip(ys, ['层 0: 1×FFT(8)', '层 1: 2×FFT(4)  偶/奇拆分',
                        '层 2: 4×FFT(2)', '层 3: 8×FFT(1)  递归基']):
    ax.text(10.2, y, lbl, ha='left', va='center', fontsize=8.2, color='#2c3e50')

ax.set_xlim(-0.2, 13.5)
ax.set_ylim(-0.2, 4.2)
ax.axis('off')
ax.set_title('N=8  my_fft(x) 递归分治树\n'
             '向下拆分（偶/奇）→ N=1 基础情形 → 向上蝶形合并',
             fontsize=10.5, fontweight='bold', pad=4)
ax.legend(handles=[mpatches.Patch(fc=COLORS[n], ec='white', lw=1.5, alpha=0.88,
                                   label=f'FFT(N={n})') for n in [8, 4, 2, 1]],
          loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.show()

## 1. 递归结构：蝶形合并

Cooley-Tukey 核心恒等式：

```
X[k]     = E[k] + exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
X[k+N/2] = E[k] - exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
```

其中 `E = fft(x[::2])`（偶数下标），`O = fft(x[1::2])`（奇数下标）。
旋转因子 `exp(-2πi·k/N)` 乘完就是「蝶形（butterfly）」操作；`X[k+N/2]` 只是符号翻转，**不需要额外乘法**。
递归基：`N==1` 时 DFT 就是原值本身，直接返回。

In [ ]:
# 演示：手算 N=2 的 DFT vs FFT 公式
x2 = np.array([3.0, 7.0])
# DFT 定义: X[k] = sum_n x[n]*exp(-2pi*i*k*n/N)
X2_def = np.array([
    x2[0] + x2[1],           # k=0: exp(0)=1
    x2[0] - x2[1],           # k=1: exp(-pi*i)=-1
], dtype=complex)
X2_np = np.fft.fft(x2)
print('手算 X[0]:', X2_def[0], '  numpy:', X2_np[0])
print('手算 X[1]:', X2_def[1], '  numpy:', X2_np[1])
print('误差:', np.max(np.abs(X2_def - X2_np)))

## 2. 必须是 2 的幂；非 2 的幂用补零（zero padding）

Cooley-Tukey 每步把 N 对半分，要求 N 是 2 的幂次（`N = 2^k`）。
对任意长度 `L` 的信号，标准做法是找最小的 `N = 2^ceil(log2(L))`，在末尾补零到 N，得到 N 点频谱。注意：补零**不会增加**物理频率分辨率（那由原始信号的时长决定），只是把频率网格插得更密，让谱线看起来更平滑。

```
N_padded = 2 ** int(np.ceil(np.log2(len(x))))
x_padded = np.zeros(N_padded, dtype=complex)
x_padded[:len(x)] = x
```

In [ ]:
# 演示：N=5 补零到 8
x5 = np.array([1, 2, 3, 4, 5], dtype=float)
N_padded = 2 ** int(np.ceil(np.log2(len(x5))))
x_pad = np.zeros(N_padded)
x_pad[:len(x5)] = x5
print(f'原始长度: {len(x5)}  补零后: {N_padded}')
print('补零后:', x_pad)
print('np.fft.fft(补零后):', np.round(np.fft.fft(x_pad), 4))

## 3. 数值误差：浮点累积 < 1e-10

递归 FFT 有 `log₂N` 层，每层做浮点乘法；误差随层数线性累积，量级约 `N·log₂N·ε_machine`。
对 `N=1024`，double 精度下误差约 `1024·10·2.2e-16 ≈ 2e-12`，远低于 `1e-10`。
验证工具：`np.testing.assert_allclose(my_result, reference, atol=1e-10)`，失败时会打印最大偏差位置。

```
# atol 是绝对容忍，rtol 是相对容忍；对复数频谱两者都要设
np.testing.assert_allclose(A, B, rtol=1e-9, atol=1e-10)
```

In [ ]:
# 演示：量化不同 N 下的浮点误差（朴素 DFT vs numpy.fft，说明误差量级）
print("     N  max_abs_err (naive_dft vs numpy.fft)")
for N in [4, 16, 64, 256, 1024, 4096]:
    x = np.random.randn(N)
    ref = np.fft.fft(x)
    n_arr = np.arange(N)
    k_arr = n_arr.reshape((N, 1))
    M_naive = np.exp(-2j * np.pi * k_arr * n_arr / N)
    naive = M_naive @ x.astype(complex)
    err = np.max(np.abs(naive - ref))  # naive O(N²) 与 numpy 的差异
    print(f"{N:>6}  {err:.2e}")
print("\n(以上为朴素 DFT 与 numpy.fft 误差；实现 my_fft 后 cell 13 将真正验证其误差 < 1e-9)")


## 4. ✏️ 实现 `my_fft(x)`（递归 Cooley-Tukey）

**五步提示**：

1. 递归基：长度为 1 时直接返回复数输入
2. 偶奇拆分：把偶数位和奇数位分成两个子问题
3. 旋转因子：`twiddle` 只负责一圈里的相位步进
4. 蝶形合并：把 `E` 和 `O` 组合成上半 / 下半频谱
5. 拼接返回：先上半，再下半

**精度要求**：`np.allclose(my_fft(x), np.fft.fft(x), atol=1e-9)` 通过

**卡住回**：L38 的蝶形图、L38/L39 的分治树，或 `solutions/L39_fft_implement_solutions.md`


In [ ]:
def my_fft(x: np.ndarray) -> np.ndarray:
    """递归 Cooley-Tukey radix-2 FFT。x 长度必须是 2 的幂。"""
    N = len(x)
    # ✏️ TODO 1: 递归基 — N==1 时直接返回
    raise NotImplementedError("TODO 1-5: 完整实现 Cooley-Tukey 递归 FFT（递归基/偶奇拆分/旋转因子/蝶形合并/拼接）")

    # ✏️ TODO 2: 递归拆分偶/奇下标
    E = ...  # my_fft(x[::2])
    O = ...  # my_fft(x[1::2])

    # ✏️ TODO 3: 旋转因子 twiddle = exp(-2pi*i * k / N), k = 0..N/2-1
    k = np.arange(N // 2)
    twiddle = ...

    # ✏️ TODO 4: 蝶形合并
    top    = ...  # E + twiddle * O
    bottom = ...  # E - twiddle * O

    # ✏️ TODO 5: 拼接返回
    return ...

In [ ]:
# ── 正确性检查 ──────────────────────────────────────────────
import numpy as np
x_demo = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
try:
    out_demo = my_fft(x_demo)
    if out_demo is None or out_demo is ...:
        print("⬜ 请先实现 my_fft 的各 TODO 项")
    else:
        ref_demo = np.fft.fft(x_demo)
        print('最大误差:', np.max(np.abs(out_demo - ref_demo)))
        all_pass = True
        for N in [4, 8, 16, 64, 256]:
            x = np.random.randn(N)
            if not np.allclose(my_fft(x), np.fft.fft(x), atol=1e-10):
                print(f'✗ N={N} 失败'); all_pass = False
        if all_pass:
            print('✅ 所有 N in [4,8,16,64,256] 通过，误差 < 1e-9')
except NotImplementedError:
    print("⬜ 请先实现 my_fft 的各 TODO 项，再运行检查格")
except TypeError:
    print("⬜ 请先替换 ... 占位符后再运行检查格")
except RecursionError:
    print("⬜ my_fft 递归异常：检查递归基（N==1 时的 return）")


In [ ]:
# ── Aurora 连接：验证 my_fft 与 aurora_fft 数值等价 ─────────────────────────────
# aurora_fft 已在 cell 2 导入，此处做最终比较
try:
    x_check = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
    out_my = my_fft(x_check)
    if out_my is None or out_my is ...:
        print("□ 请先实现 my_fft 后再运行此格")
    else:
        out_aurora = aurora_fft(x_check)
        max_diff = float(np.max(np.abs(out_my - out_aurora)))
        np.testing.assert_allclose(out_my, out_aurora, atol=1e-10,
            err_msg="my_fft 与 aurora_fft 输出不一致，请检查蝶形合并逻辑")
        print(f"✅ my_fft ↔ aurora_fft 最大差异: {max_diff:.2e}  (< 1e-9)")
        print("   两者数值等价，算法路径不同（递归 vs 迭代 Cooley-Tukey）")
except NotImplementedError:
    print("□ 请先实现 my_fft 后再运行此格")
except TypeError:
    print("□ 请先替换 ... 占位符后再运行")
except RecursionError:
    print("□ my_fft 递归异常：检查递归基（N==1 时的 return）")


## 补充 · 非 2 幂、补零与 STFT 调用约定

本课的 `my_fft(x)` 只要求处理 2 的幂长度，这是为了让递归分治保持清楚。

生产代码里也有同样的边界：

- `aurora.audio.transforms.fft(x)`：输入长度必须是 2 的幂；否则抛出 `ValueError`
- `aurora.audio.transforms.next_power_of_two(n)`：返回不小于 `n` 的最小 2 的幂
- `aurora.audio.stft.stft(...)`：在每帧送进 `rfft/fft` 前，会把 `n_fft` 防护到 2 的幂长度

所以你可以把约定记成：

```text
普通 FFT：你负责给它 2 的幂长度
STFT 流水线：每帧会按 n_fft/next_power_of_two 做补零防护
```

这会在 L44 手写 STFT 时再次出现：`win_len` / `n_fft` 通常取 256、512、1024 这类 2 的幂。


In [ ]:
import numpy as np
from aurora.audio.transforms import fft, next_power_of_two

x5 = np.ones(5)
try:
    fft(x5)
except ValueError as exc:
    print("非 2 幂直接 fft 会失败：", str(exc).split(".")[0])

target = next_power_of_two(len(x5))
x_pad = np.pad(x5, (0, target - len(x5)))
X_pad = fft(x_pad)

assert target == 8
assert x_pad.shape == (8,)
assert X_pad.shape == (8,)
print(f"✅ len=5 → 补零到 {target} 后可运行；这正是 L44/STFT 会用到的边界意识")


## 5. 参数实验：速度对比

- `N = 512`：naive DFT（O(N²)) vs `my_fft`（O(N log N)）vs `np.fft.fft`（C 扩展）
- 预期：`my_fft` 比 naive DFT 快约 `log₂(512) = 9` 倍；`np.fft.fft` 比 `my_fft` 快 10-50 倍（C 扩展 + SIMD）
- 调整 `N` 观察：倍数比例随 `N` 增大而增大（log 增益更显著）

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """暴力 O(N²) DFT，用于速度基准对比。"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return M @ x.astype(complex)

N = 512
x_bench = np.random.randn(N)

reps = 20
t0 = time.perf_counter()
for _ in range(reps): naive_dft(x_bench)
t_naive = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): my_fft(x_bench)
t_my = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): np.fft.fft(x_bench)
t_np = (time.perf_counter() - t0) / reps * 1000

print(f'N={N}  (平均 {reps} 次，单位 ms)')
print(f'  naive_dft : {t_naive:8.3f} ms')
print(f'  my_fft    : {t_my:8.3f} ms   ({t_naive/t_my:.1f}× faster than naive)')
print(f'  np.fft.fft: {t_np:8.3f} ms   ({t_my/t_np:.1f}× faster than my_fft)')
print(f'\n理论加速比 naive→FFT: log₂({N}) = {np.log2(N):.1f}')

---
→ **完成 `my_fft` 实现后，打开 [L42 · FFT 图形化](L42_visual_fft.ipynb)，对照蝴蝶图与各类信号的频谱形态，检验输出是否符合直觉，再回来。**

## 本课收束

`my_fft(x)` 通过递归蝶形操作输出 N 个复数频谱系数，误差控制在 `1e-9` 以内，与 `aurora.audio.transforms.fft` 在数值上等价，但算法路径独立：生产版采用**迭代式** Cooley-Tukey（位逆序置换 + 非递归蝴蝶），L41 将实现该迭代版本；两者都能正确驱动 STFT 的内层计算。下一课 L40 将在复数频谱上分析幅度谱与相位谱，验证频率分辨率与信号长度的关系。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：FFT 手算与递归结构（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`my_fft` 递归的基础情形是什么？  
（N=? 时直接返回，返回的是什么？）

**问 2**：对 N=8 的输入，递归树有几层？每层共有多少个蝶形运算？

**问 3**：`x = [1, 0, -1, 0]`，用 Cooley-Tukey 分治手算 X[1]：
- x_even = x[0,2] = [1, -1]，E = DFT([1,-1]) = ?
- x_odd  = x[1,3] = [0, 0]，O = DFT([0, 0])  = ?
- W_4^1 = e^{-πi/2} = ?
- X[1] = E[1] + W_4^1 · O[1] = ?

**问 4**：`my_fft(np.array([1,1,1,1]))` 的结果是什么？  
（提示：均匀信号的 DFT 只在 k=0 有非零值）

**问 5**：为什么 `my_fft` 要求输入长度是 2 的幂？不满足时如何解决？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：基础情形 N=1
x1 = np.array([3.7])
X1_ref = np.fft.fft(x1)
try:
    X1 = my_fft(x1)
    assert np.allclose(X1, X1_ref, atol=1e-12)
    print(f"Q1 ✅  N=1 基础情形：my_fft([3.7])={X1}（返回输入本身作为复数）")
except NotImplementedError:
    print("⬜ Q1：请先实现 my_fft()，再运行对答案格")

# 问2：N=8 树结构
N8 = 8
n_layers = int(np.log2(N8))
butterflies_per_layer = N8 // 2
total_butterflies = butterflies_per_layer * n_layers
assert n_layers == 3 and butterflies_per_layer == 4 and total_butterflies == 12
print(f"Q2 ✅  N=8: {n_layers} 层，每层 {butterflies_per_layer} 个蝶形，共 {total_butterflies} 个蝶形")

# 问3：x=[1,0,-1,0] 手算 X[1]
x4 = np.array([1.0, 0.0, -1.0, 0.0])
X4_ref = np.fft.fft(x4)
# E = DFT([1,-1]) = [0, 2]
E4 = np.fft.fft(np.array([1.0, -1.0]))
O4 = np.fft.fft(np.array([0.0, 0.0]))
W41 = np.exp(-2j * np.pi * 1 / 4)
X1_calc = E4[1] + W41 * O4[1]
assert np.isclose(X1_calc, X4_ref[1], atol=1e-12)
print(f"Q3 ✅  E[1]={E4[1]:.4f}, O[1]={O4[1]:.4f}, W_4^1={W41:.4f}")
print(f"      X[1] = {E4[1]:.4f} + {W41:.4f}×{O4[1]:.4f} = {X1_calc:.4f}")
try:
    X4_mine = my_fft(x4)
    assert np.allclose(X4_mine, X4_ref, atol=1e-9)
    print(f"      my_fft 验证：{np.round(X4_mine, 4)}  ✅")
except NotImplementedError:
    print("      ⬜ 请先实现 my_fft()，再运行对答案格")

# 问4：x=[1,1,1,1]
x_dc = np.array([1.0, 1.0, 1.0, 1.0])
X_dc_ref = np.fft.fft(x_dc)
assert np.isclose(X_dc_ref[0], 4+0j, atol=1e-12)
assert np.allclose(X_dc_ref[1:], 0+0j, atol=1e-12)
print(f"Q4 ✅  my_fft([1,1,1,1]) = {np.round(X_dc_ref, 4)}（只有 k=0 处=4，其余=0）")

# 问5：2的幂 — 非2的幂需补零
x5 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
N_padded = 2 ** int(np.ceil(np.log2(len(x5))))
assert N_padded == 8
x5_padded = np.pad(x5, (0, N_padded - len(x5)))
X5_padded = np.fft.fft(x5_padded)
print(f"Q5 ✅  N=5 不是2的幂，补零至 N={N_padded}；频谱长度={len(X5_padded)}（插值频率轴）")
print("\n🎉 FFT 手算挑战通过！递归 Cooley-Tukey 结构已内化。")

In [ ]:
# ✏️ 本课自评
l39_review = {
    "cooley_tukey_recursion":  None,  # 记住递归结构：base N=1，split even/odd，butterfly？True/False
    "my_fft_implemented":      None,  # my_fft 实现并通过 atol=1e-9 验证？True/False
    "error_bound_understood":  None,  # 理解浮点误差 < 1e-9 的来源（log N层累积）？True/False
    "speedup_quantified":      None,  # 知道 N=1024 时 FFT 比 DFT 快约 205 倍？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
    "stft_padding_contract":   None,  # 理解 fft 要 2 的幂、STFT 会补零防护？True/False
}

unfilled = [k for k, v in l39_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l39_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L39 全部通关！进入 L40：用频谱解释真实信号')

---

→ **下一课**　[L40 · 频谱分析实战](L40_spectrum.ipynb)

> 下节课将学习 **频谱分析实战**：幅度谱 / 相位谱 / 频率分辨率，440Hz+880Hz 混合信号。